# Lab 4 — Reading and Writing Data Files

**Course:** Python for AI & Data  
**Analyst:** BeanScene Analytics Team  
**Date:** 2024-01-29

---

## The BeanScene Scenario

Load BeanScene's menu from CSV, read performance thresholds from JSON, run the analysis, and save results to disk.


---
## Task 1 — Confirm Environment


In [ ]:
import os
from pathlib import Path

print("Working directory:", os.getcwd())

RAW_DIR = Path("data") / "raw"
PROCESSED_DIR = Path("data") / "processed"

menu_path = RAW_DIR / "beanscene_menu.csv"
config_path = RAW_DIR / "beanscene_config.json"

print("Menu CSV exists:", menu_path.exists())
print("Config JSON exists:", config_path.exists())


---
## Task 2 — Load Menu from CSV


In [ ]:
import csv

menu = []
with open(menu_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        menu.append({
            "name": row["name"],
            "price": float(row["price"]),
            "units_sold": int(row["units_sold"]),
            "category": row["category"],
        })

print(menu[0])
print("price type:", type(menu[0]["price"]))
print("units_sold type:", type(menu[0]["units_sold"]))
print(f"Total items loaded: {len(menu)}")


---
## Task 3 — Load Config from JSON


In [ ]:
import json

with open(config_path, encoding="utf-8") as f:
    config = json.load(f)

print("Café name:", config["cafe_name"])
print("High threshold:", config["performance_thresholds"]["high"])
print("Low threshold:", config["low_performer_threshold"])


---
## Task 4 — Analyse the Menu


In [ ]:
high_threshold = config["performance_thresholds"]["high"]
low_threshold = config["low_performer_threshold"]

results = []

for item in menu:
    revenue = item["price"] * item["units_sold"]
    if revenue >= high_threshold:
        tier = "High"
    elif revenue >= low_threshold:
        tier = "Medium"
    else:
        tier = "Low"
    results.append({"name": item["name"], "revenue": round(revenue, 2), "tier": tier})

for r in results:
    print(f"{r['name']:<16}: ${r['revenue']:>7.2f}  [{r['tier']}]")


---
## Task 5 — Save Results to CSV


In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_csv = PROCESSED_DIR / "menu_results.csv"

with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "revenue", "tier"])
    writer.writeheader()
    writer.writerows(results)

print("Saved:", output_csv)


---
## Task 6 — Save Summary to JSON


In [ ]:
total_revenue = sum(r["revenue"] for r in results)
best_seller = max(results, key=lambda r: r["revenue"])["name"]

summary = {
    "cafe_name": config["cafe_name"],
    "week": config["analysis_week"],
    "total_revenue": round(total_revenue, 2),
    "items_analysed": len(results),
    "best_seller": best_seller,
}

output_json = PROCESSED_DIR / "weekly_summary.json"
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:", output_json)


---
## Task 7 — Verify Outputs


In [ ]:
print("=== menu_results.csv ===")
with open(output_csv, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(row)

print("\n=== weekly_summary.json ===")
with open(output_json, encoding="utf-8") as f:
    loaded_summary = json.load(f)
print(json.dumps(loaded_summary, indent=2))


---
## ⭐ Optional Advanced Tasks — Solutions


In [ ]:
# Optional 1 — Low Performers File
low_performers = [r for r in results if r["tier"] == "Low"]
low_path = PROCESSED_DIR / "low_performers.csv"

with open(low_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "revenue", "tier"])
    writer.writeheader()
    writer.writerows(low_performers)

# Append a footer row with the count
with open(low_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["TOTAL LOW PERFORMERS", "", len(low_performers)])

print(f"Saved {len(low_performers)} low performers to {low_path}")


In [ ]:
# Optional 3 — Category Summary
category_totals = {}

for item in menu:
    cat = item["category"]
    revenue = item["price"] * item["units_sold"]
    if cat not in category_totals:
        category_totals[cat] = 0
    category_totals[cat] += revenue

# Round all values
category_totals = {k: round(v, 2) for k, v in category_totals.items()}

cat_path = PROCESSED_DIR / "category_summary.json"
with open(cat_path, "w", encoding="utf-8") as f:
    json.dump(category_totals, f, indent=2)

print(json.dumps(category_totals, indent=2))
